# 03 — Inhibición recurrente con células de Renshaw

## Objetivo

Construir un circuito neuromotor mínimo con un pool de motoneuronas (MN), un pool pequeño de células de Renshaw (R), excitación **MN → R** e inhibición recurrente **R → MN**.

Las motoneuronas usan inicialmente el preset *Regular Spiking*. Las Renshaw usan *Fast Spiking* como **aproximación funcional** para representar una interneurona rápida; no es una calibración biológica específica ni permite inferencias cuantitativas sobre células de Renshaw reales.

## Pregunta central y rol de la semilla

Esta notebook pregunta: **¿puede un grupo de interneuronas activado por las motoneuronas devolver una corriente que limite su actividad posterior?**

La semilla fija, en un mismo orden reproducible:

1. heterogeneidad de las motoneuronas;
2. heterogeneidad de las Renshaw;
3. conexiones `MN → R`;
4. conexiones `R → MN`;
5. ruido de ambos pools.

Así, activar o desactivar la inhibición con la misma semilla compara el mismo circuito y el mismo ruido. La semilla 42 no es “mejor”: es solo una realización reproducible.

### Flujo causal del circuito

In [4]:
from pathlib import Path
import sys

# Funciona si el kernel inicia en la raíz o dentro de notebooks/.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

from src.metrics import force_summary, neural_summary, population_activity
from src.muscle import normalize_force, spikes_to_force
from src.renshaw import simulate_renshaw_circuit
from src.visualization import animate_muscle_contraction, plot_raster, plot_renshaw_circuit, spike_times_from_matrix


In [ ]:
from src.visualization import plot_renshaw_signal_flow

plot_renshaw_signal_flow()
plt.show()

## Convención de pesos y corrientes

Las filas de cada matriz son neuronas de origen y las columnas son neuronas de destino. `W_MN_to_R` y `W_R_to_MN` guardan **magnitudes positivas**. La segunda matriz es inhibitoria por su uso en la ecuación, no por su signo:

$
I_{MN}=I_{motor}+I_{ruido}-I_{Renshaw},\qquad
I_R=I_{MN	o R}+I_{ruido,R}.
$

Cada spike incrementa una variable sináptica que decae exponencialmente con `tau_syn`. La integración neuronal es Euler y el reset se aplica cuando `v ≥ 30 mV`.

## Las dos matrices de Renshaw, paso a paso

Con 20 motoneuronas y 5 células de Renshaw:

- `W_MN_to_R.shape == (20, 5)`
- `W_R_to_MN.shape == (5, 20)`

Siempre se usa **fila = origen** y **columna = destino**.

### `W_MN_to_R`

`W_MN_to_R[i, j]` indica cuánto excita la motoneurona `i` a la Renshaw `j`. Sus valores son magnitudes positivas.

### `W_R_to_MN`

`W_R_to_MN[k, i]` indica la magnitud con que la Renshaw `k` inhibe a la motoneurona `i`. También se almacena positiva, pero el modelo la resta:

\[
I_{MN,i}=I_{motor}+I_{ruido,i}-\sum_k W_{R\to MN}[k,i]S_{R,k}
\]

Ejemplo: si una Renshaw tiene estado `S=2` y su peso hacia una MN es `1.5`, aporta una magnitud inhibitoria `3.0`, que se resta del input de esa MN.

Esta convención evita mezclar pesos negativos con una segunda resta. El signo biológico está en la ecuación, no en el valor guardado en la matriz.

In [5]:
def plot_renshaw_results(results, bin_size_ms=20):
    """Muestra arquitectura, matrices y las señales pedidas en un panel."""
    t = results["time"]
    dt = results["dt"]
    mn_times = spike_times_from_matrix(results["spikes_MN"], t)
    r_times = spike_times_from_matrix(results["spikes_R"], t)
    bins_mn, activity_mn = population_activity(results["spikes_MN"], dt, bin_size_ms)
    bins_r, activity_r = population_activity(results["spikes_R"], dt, bin_size_ms)
    fig, axes = plt.subplots(3, 3, figsize=(17, 13))
    plot_renshaw_circuit(results["W_MN_to_R"], results["W_R_to_MN"], axes[0, 0])
    for ax, matrix, title in [(axes[0,1], results["W_MN_to_R"], "W_MN_to_R (+)"),
                              (axes[0,2], results["W_R_to_MN"], "W_R_to_MN (magnitud +)")]:
        im = ax.imshow(matrix, aspect="auto", cmap="viridis", origin="lower")
        ax.set(title=title, xlabel="Destino", ylabel="Origen"); fig.colorbar(im, ax=ax, shrink=.75)
    plot_raster(mn_times, axes[1,0]); axes[1,0].set_title("Raster de motoneuronas")
    plot_raster(r_times, axes[1,1]); axes[1,1].set_title("Raster de Renshaw")
    axes[1,2].plot(t, results["I_R_to_MN"].mean(axis=0), color="tab:red")
    axes[1,2].set(title="Inhibición media recibida por MN", xlabel="Tiempo (ms)", ylabel="Corriente (u.a.)")
    axes[2,0].plot(t, results["V_MN"][0]); axes[2,0].set_title("v de MN representativa")
    axes[2,1].plot(t, results["V_R"][0], color="tab:orange"); axes[2,1].set_title("v de Renshaw representativa")
    axes[2,2].step(bins_mn, activity_mn, where="post", label="MN")
    axes[2,2].step(bins_r, activity_r, where="post", label="Renshaw")
    axes[2,2].set(title=f"Actividad poblacional ({bin_size_ms} ms)", xlabel="Tiempo (ms)", ylabel="Spikes/bin"); axes[2,2].legend()
    for ax in axes.flat: ax.grid(alpha=.2)
    fig.tight_layout(); plt.show()


## Cómo leer el panel de resultados

1. **Diagrama**: muestra qué conexiones existen, no cuándo están activas.
2. **Matrices**: permiten localizar origen, destino y magnitud.
3. **Raster MN y R**: comprueba la secuencia esperada: actividad MN, activación Renshaw y modificación posterior de MN.
4. **Voltajes representativos**: muestran la dinámica individual, pero una neurona no resume todo el pool.
5. **Actividad poblacional**: suma spikes por bin; depende del tamaño del bin.
6. **Corriente inhibitoria media**: promedio entre todas las MN y todo el tiempo. Incluye períodos de reposo, por lo que no equivale al máximo feedback durante el estímulo.

Una inhibición mayor puede reducir spikes sin silenciar todas las neuronas. El resultado debe describirse como comportamiento de este modelo, no como medición fisiológica.

## GUI

Cada fila define brevemente el parámetro antes de mostrar su control. Los límites pequeños protegen la velocidad y legibilidad de las figuras.

In [6]:
def described_widget(text, widget):
    label = widgets.HTML(f"<div style='width:310px;font-size:13px'>{text}</div>")
    return widgets.HBox([label, widget])

controls_spec = {
    "n_motor": ("<b>Motoneuronas</b><br>Tamaño del pool motor (2–30).", widgets.IntSlider(value=20, min=2, max=30)),
    "n_renshaw": ("<b>Células de Renshaw</b><br>Tamaño del pool inhibitorio (1–10).", widgets.IntSlider(value=5, min=1, max=10)),
    "amplitude": ("<b>Comando motor</b><br>Amplitud del input externo común.", widgets.FloatSlider(value=12, min=0, max=30, step=.5)),
    "input_type": ("<b>Tipo de input</b><br>Forma temporal del comando.", widgets.Dropdown(options=["constant","pulse","ramp","sinusoidal","motor_plan"], value="motor_plan")),
    "parameter_noise": ("<b>Ruido de parámetros</b><br>Heterogeneidad relativa de a, b, c y d.", widgets.FloatSlider(value=.03, min=0, max=.2, step=.01)),
    "input_noise": ("<b>Ruido de input</b><br>Desvío del ruido individual de corriente.", widgets.FloatSlider(value=.5, min=0, max=5, step=.1)),
    "p_mn_to_r": ("<b>Probabilidad MN → R</b><br>Densidad de conexiones excitatorias.", widgets.FloatSlider(value=.5, min=0, max=1, step=.05)),
    "w_mn_to_r": ("<b>Peso MN → R</b><br>Magnitud de excitación por conexión.", widgets.FloatSlider(value=1.5, min=0, max=5, step=.1)),
    "p_r_to_mn": ("<b>Probabilidad R → MN</b><br>Densidad del feedback inhibitorio.", widgets.FloatSlider(value=.6, min=0, max=1, step=.05)),
    "w_r_to_mn": ("<b>Peso R → MN</b><br>Magnitud que se resta al input motor.", widgets.FloatSlider(value=1., min=0, max=8, step=.1)),
    "synaptic_tau": ("<b>Constante sináptica</b><br>Decaimiento exponencial en ms.", widgets.FloatSlider(value=10, min=1, max=50, step=1)),
    "total_time": ("<b>Duración</b><br>Tiempo total de simulación (ms).", widgets.IntSlider(value=1000, min=300, max=2000, step=100)),
    "dt": ("<b>dt</b><br>Paso de Euler en ms.", widgets.FloatSlider(value=.5, min=.1, max=1, step=.1)),
    "seed": ("<b>Semilla</b><br>Permite repetir conectividad y ruido.", widgets.IntSlider(value=42, min=0, max=999)),
    "recurrent_inhibition": ("<b>Inhibición recurrente</b><br>Activa o desconecta R → MN.", widgets.Checkbox(value=True)),
}
output = widgets.Output()
button = widgets.Button(description="Simular circuito", button_style="success")

def run_gui(_=None):
    with output:
        clear_output(wait=True)
        kwargs = {name: item[1].value for name, item in controls_spec.items()}
        results = simulate_renshaw_circuit(**kwargs)
        print(f"Spikes MN: {results['spikes_MN'].sum()} | Spikes R: {results['spikes_R'].sum()}")
        plot_renshaw_results(results)

button.on_click(run_gui)
display(widgets.VBox([widgets.HTML("<h3>Controles del circuito MN–Renshaw</h3>"),
                      *[described_widget(text, widget) for text, widget in controls_spec.values()], button]), output)


Output()

## Supuestos y límites

- Los pesos y unidades de corriente son abstractos; no están ajustados a datos experimentales.
- Las conexiones son Bernoulli aleatorias y no incluyen retardos ni plasticidad.
- RS y FS son proxies funcionales. El circuito permite estudiar comportamiento cualitativo, no reproducir exactamente un circuito espinal.